# Phase 1: Demand Prediction (XGBoost ML)

Train an XGBoost model to predict historical sales based on inventory features.

## Import Required Libraries

In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import pickle

## Load and Preprocess Data

In [35]:
# Load the dataset
df = pd.read_csv('inventory_data.csv')

# Features and target
features = ['QuantityInStock', 'DaysUntilExpiry', 'ProductCategory']
target = 'HistoricalSales'

# One-hot encode ProductCategory
encoder = OneHotEncoder(sparse_output=False, drop='first')  # drop first to avoid multicollinearity
encoded_categories = encoder.fit_transform(df[['ProductCategory']])
encoded_df = pd.DataFrame(encoded_categories, columns=encoder.get_feature_names_out(['ProductCategory']))

# Combine features
X = pd.concat([df[['QuantityInStock', 'DaysUntilExpiry']], encoded_df], axis=1)
y = df[target]

# Split into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Train XGBoost Model

In [36]:
# Initialize XGBoost Regressor with specified hyperparameters
model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate using Mean Absolute Error
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error on test set: {mae:.2f}")

Mean Absolute Error on test set: 18.34


## Save Model and Encoder

In [37]:
# Save the trained model
with open('demand_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Also save the encoder for later use in testing
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

print("Model trained and saved as 'demand_model.pkl'")
print("Encoder saved as 'encoder.pkl'")

Model trained and saved as 'demand_model.pkl'
Encoder saved as 'encoder.pkl'
